# Distributional Stage-1

Does predicting *how ratings are spread* beat predicting only their mean?

| mediator | width | Stage-1 predicts |
|---|---|---|
| `Hyb 7` | 7 | per-emotion population mean |
| `Hyb_sd 14` | 14 | mean (7) + across-rater std (7) |
| `Hyb_hist 35` | 35 | 7 emotions x 5 rating bins |

Everything else is held fixed: same ridge Stage-1, same anchor C, same
validation-group hyperparameter selection. Read from
`output/raw_all_final.csv`, which contains only runs computed after both
the ratings.csv repair and the feature normalisation.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from scipy.stats import wilcoxon

SRC = '../output/raw_all_final.csv'
UNIT = ['fold', 'domain', 'user_id']

# raw mediator name -> column label in the table
COLS = {'population': 'pop',
        'identity': 'Direct',
        'emotion': 'Hyb 7',
        'emotion_sd': 'Hyb_sd 14',
        'emotion_hist': 'Hyb_hist 35'}

LABEL = {'clip': 'CLIP frozen', 'qwen8b': 'Qwen3-VL 8B',
         'clip_ft': 'CLIP-ft (score)', 'clip_ft_emo': 'CLIP-ft (emotion)',
         'qwen4b': 'Qwen3-VL 4B'}

raw = pd.read_csv(SRC, low_memory=False)
print(f'{len(raw):,} rows | backbones: {sorted(raw.backbone.unique())}')


56,502 rows | backbones: ['clip']


## Build the table


In [2]:
def build(backbone, variant='C', head='ridge'):
    """One row per support size. None if this backbone has no
    distribution run yet -- reported rather than silently skipped."""
    d = raw[(raw.backbone == backbone) & (raw.variant == variant)
            & (raw['head'] == head) & raw.mediator.isin(COLS)]
    rows = []
    for n in (10, 25, 50, 100):
        g = d[d.n_train == n]
        if g.empty:
            continue
        g = g.groupby(UNIT + ['mediator'], as_index=False)['srocc'].mean()
        p = g.pivot_table(index=UNIT, columns='mediator', values='srocc')
        if not set(COLS) <= set(p.columns):
            continue
        p = p.dropna(subset=list(COLS))
        row = {'n_train': n}
        for m, label in COLS.items():
            row[label] = p[m].mean()
        row['hist vs Hyb7 p'] = wilcoxon(p['emotion_hist'], p['emotion'])[1]
        rows.append(row)
    return pd.DataFrame(rows).set_index('n_train') if rows else None


## Colours

Background colour goes on the **column header cells only**. The numbers
themselves stay plain, so nothing competes with reading them.


In [3]:
# --------------------------------------------------------------
# Which header gets which background. Move a name between these two
# lists to change its colour; reorder a list to reorder the columns.
# --------------------------------------------------------------
GREY_HEADERS = ['pop', 'Direct']
BLUE_HEADERS = ['Hyb 7', 'Hyb_sd 14', 'Hyb_hist 35']

GREY_BG, BLUE_BG = '#37474f', '#1565c0'


def show(df, caption):
    from IPython.display import display

    order = ([c for c in GREY_HEADERS + BLUE_HEADERS if c in df.columns]
             + [c for c in df.columns
                if c not in GREY_HEADERS + BLUE_HEADERS])
    df = df[order]

    styles = [
        {'selector': 'caption',
         'props': [('caption-side', 'top'), ('font-size', '106%'),
                   ('font-weight', '700'), ('padding-bottom', '8px'),
                   ('text-align', 'left')]},
        {'selector': 'th.col_heading',
         'props': [('padding', '6px 10px'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'right'),
                                     ('padding', '4px 10px')]},
    ]

    # background on the header cell of each column, by position
    for i, c in enumerate(df.columns):
        if c in GREY_HEADERS:
            bg, fg = GREY_BG, 'white'
        elif c in BLUE_HEADERS:
            bg, fg = BLUE_BG, 'white'
        else:
            bg, fg = '#eceff1', '#546e7a'
        styles.append({'selector': f'th.col_heading.col{i}',
                       'props': [('background-color', bg), ('color', fg),
                                 ('font-weight', '600')]})

    fmt = {c: '{:.4f}' for c in df.columns}
    return display(df.style.format(fmt)
                   .set_caption(caption)
                   .set_table_styles(styles))


## CLIP frozen


In [4]:
t = build('clip')
show(t, 'CLIP frozen | anchor C | ridge | 3 seeds | 387 user-domain units')


,pop,Direct,Hyb 7,Hyb_sd 14,Hyb_hist 35,hist vs Hyb7 p
n_train,,,,,,
10,0.4159,0.4196,0.4138,0.4185,0.4128,0.2554
25,0.4159,0.4312,0.4152,0.4192,0.4247,0.0001
50,0.4159,0.4404,0.4232,0.4285,0.4347,0.0000
100,0.4159,0.4559,0.4352,0.4343,0.4420,0.0005


## Other backbones

Filled in automatically once their anchor-C distribution runs land in
`raw_all_final.csv`. Rerun the notebook; no code change needed.


In [5]:
for bb in ('qwen8b', 'clip_ft', 'clip_ft_emo', 'qwen4b'):
    t = build(bb)
    if t is None:
        print(f'{LABEL[bb]}: not run yet')
    else:
        show(t, f'{LABEL[bb]} | anchor C | ridge | 3 seeds')


Qwen3-VL 8B: not run yet
CLIP-ft (score): not run yet
CLIP-ft (emotion): not run yet
Qwen3-VL 4B: not run yet
